# nested-param-group-loop — worked example 2: manual SGD with per-group momentum

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `nested-param-group-loop`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The nested `param_groups -> params` loop carries any per-group hyperparameter through to each parameter. With momentum, each group reads its own `lr` and `momentum`, and each parameter keeps a velocity buffer that is updated then applied.

## Worked solution

We construct two groups with different `lr` and `momentum`, attach fake gradients, and keep a per-parameter velocity buffer in a dict keyed by `id(p)`. The outer loop reads both `lr` and `momentum` from the group dict. The inner loop skips `grad is None`, then updates the velocity `v = momentum * v + grad` and applies `p -= lr * v` in place. Tracking the buffer outside the params mirrors how real optimizers stash state per parameter. We seed, run a single step (so velocity equals the raw gradient on the first call), and print the resulting velocity-driven move for one parameter.

In [ ]:
import torch as t

t.manual_seed(1)

w = t.nn.Parameter(t.randn(4))
h = t.nn.Parameter(t.randn(4))
opt = t.optim.SGD([
    {'params': [w], 'lr': 0.1, 'momentum': 0.9},
    {'params': [h], 'lr': 0.2, 'momentum': 0.0},
])
w.grad = t.ones(4)
h.grad = t.full((4,), 3.0)

velocity = {}

def manual_sgd_momentum_step(optimizer):
    for group in optimizer.param_groups:
        lr = group['lr']
        mom = group['momentum']
        for p in group['params']:
            if p.grad is None:
                continue
            v = velocity.get(id(p), t.zeros_like(p.data))
            v = mom * v + p.grad
            velocity[id(p)] = v
            p.data.add_(v, alpha=-lr)

before_w = w.data.clone()
manual_sgd_momentum_step(opt)
print('w delta:', (w.data - before_w))  # first step: -lr * grad = -0.1